<a href="https://colab.research.google.com/github/siddumais/starter-notebook/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task


**Lane: refresh / content-opportunity scoring.** Skills loaded: `framing-ml-problems`, `flyrank/flyrank-data`.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring, which one, and why?*

**Task type: Ranking / scoring.**

My lane's question is *"which content items should an editor look at first for a refresh?"*, not "is this one declining, yes/no" (classification) and not "what natural groups of content exist" (clustering). "Which ones first" is exactly the ranking/scoring row in the framing table: target = a priority score, typical metric = precision@K.

Working through the four framing questions:

1. **Decision it improves:** which content items go to the top of the weekly refresh queue.
2. **Who acts, and how:** a content editor/strategist with fixed weekly capacity, they can rework a handful of pieces a week, not all of them, so the *order* of the list is the whole product.
3. **Cost of a wrong call:** refreshing a piece that wouldn't have improved wastes editor hours that a genuinely worth it piece could have used instead; missing a piece with real upside means recoverable traffic keeps leaking away unnoticed.
4. **Why data/ML, not a plain rule:** see Section 5, the individual signals that make something "worth refreshing" barely agree with each other, so no single threshold captures it.

In [7]:
import pandas as pd
import numpy as np
from google.colab import files

uploaded = files.upload()  # opens a file picker, upload content_refresh_anonymized.csv
fname = next(iter(uploaded))  # grabs whatever filename you picked

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 160)

df = pd.read_csv(fname)
print(f"{len(df):,} rows x {df.shape[1]} columns")
print(f"{df.client_id.nunique()} clients, {df.content_id.nunique()} unique content_id")
print(f"content_type breakdown:\n{df.content_type.value_counts()}")

Saving content_refresh_anonymized.csv to content_refresh_anonymized.csv
30,000 rows x 44 columns
32 clients, 30000 unique content_id
content_type breakdown:
content_type
keyword article       27207
feedly article         2096
comparison article      697
Name: count, dtype: int64


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**This is a proxy, not an observed label yet, flagging that honestly up front.**

The real, observed target for this lane would be: *did pageviews/clicks actually go up after this specific piece was refreshed?* That needs knowing which pieces were refreshed, when, and their performance just before vs. just after that event. The starter CSV is a single 90-day snapshot per content item — no history of matched refresh-then-outcome pairs, so that observed target isn't available from this file. (The warehouse's `fact_content_daily_performance` panel spans ~17 months, which *would* support building real pre/post windows around actual update dates, but wiring that up safely, without leaking future data across the window, is a data-contract-and-leakage job for later weeks, not this notebook.)

So for framing purposes today, the target is a **defined proxy priority score**, built only from signals that are already observed in this file, deliberately *not* `trend_direction` / `trend_pct`, which the data skill flags as derived and off-limits as a feature (the "label trap"). I don't want my own proxy silently re-importing someone else's rule. Three ingredients, each a real column:

- **Recent movement** — `clicks_last_30d` vs. `clicks_prev_30d`, a same-content comparison of the two 30-day windows already sitting in the data (independent of `trend_pct`; Section 5 shows just how independent).
- **Room to climb** — `avg_position` between 4 and 20, excluding `avg_position == 0` which the data dictionary says means *no data*, not rank zero. Position 1–3 is already winning; 20+ is a much longer climb.
- **Staleness** — `days_since_last_update`.

I rank each ingredient within its own distribution (0–1) and average the ranks, skipping whatever isn't observed for a given row rather than filling zeros (per the data skill's missingness warning) into an `opportunity_score`. Calling it what it is: a heuristic proxy good enough to sanity-check the shape of the problem — not a claim that it predicts real post-refresh outcomes.

**A gotcha I hit while building this, worth naming:** my first pass averaged the three ranks with `skipna=True` and let `avg_position == 0` rows (the data dictionary's "no data" code, not rank zero) into the pool anyway. Result: rows with *zero clicks and no position data at all* floated to the very top of the queue, because "staleness" was the only signal they had and averaging over one value looks the same as averaging over three good ones. That's the missingness gotcha from the data skill showing up inside a derived score, not just a raw column — so the fix is the same fix: drop `avg_position == 0` rows from the candidate pool entirely (they aren't "ranking but declining," they're a different, out-of-scope problem), rather than let a lonely signal fill in for the missing ones.

In [8]:
# avg_position == 0 means "no data" (per the data dictionary), not rank zero - these rows
# aren't "ranking but declining", they're a different problem, so they don't enter this queue at all.
lane = df[df['avg_position'] > 0].copy()
print(f"dropped {(df['avg_position'] == 0).sum()} rows with no position data; "
      f"{len(lane):,} remain as real refresh candidates")

# ingredient 1: recent movement, independent of trend_pct/trend_direction
has_prev = lane['clicks_prev_30d'] > 0
lane['recent_pct_change'] = np.where(
    has_prev,
    (lane['clicks_last_30d'] - lane['clicks_prev_30d']) / lane['clicks_prev_30d'],
    np.nan,
)
lane['has_recent_change'] = has_prev  # flag instead of a blind fillna(0) - per the data skill

#  ingredient 2: room to climb (always observed now - avg_position > 0 by construction)
lane['climb_room'] = ((lane['avg_position'] >= 4) & (lane['avg_position'] <= 20)).astype(float)

#  ingredient 3: staleness (always observed)
lane['staleness'] = lane['days_since_last_update']

def rank01(s):
    return s.rank(pct=True)

lane['rank_recent'] = rank01(-lane['recent_pct_change'])  # bigger drop -> higher rank
lane['rank_climb']  = rank01(lane['climb_room'])
lane['rank_stale']  = rank01(lane['staleness'])

signal_cols = ['rank_recent', 'rank_climb', 'rank_stale']
lane['opportunity_score']  = lane[signal_cols].mean(axis=1, skipna=True)
lane['n_signals_observed'] = lane[signal_cols].notna().sum(axis=1)  # now always >= 2

print()
print('signals observed per row (2 = no prior clicks to compare; 3 = full data):')
print(lane['n_signals_observed'].value_counts())
print()
print(lane['opportunity_score'].describe())

dropped 1205 rows with no position data; 28,795 remain as real refresh candidates

signals observed per row (2 = no prior clicks to compare; 3 = full data):
n_signals_observed
2    17036
3    11759
Name: count, dtype: int64

count    28795.000000
mean         0.494905
std          0.163453
min          0.067570
25%          0.378130
50%          0.502492
75%          0.626911
max          0.856168
Name: opportunity_score, dtype: float64


## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Metric: precision@K** — the row the framing table names for ranking/scoring, and the one that matches how an editor actually judges the tool: *"of the top K items I was told to refresh this week, how many turned out to be real wins?"* — not a global accuracy score over all 30,000 items, almost none of which will ever be looked at.

I can't compute *true* precision@K today: it needs the observed post-refresh outcome discussed in Section 2, which this snapshot doesn't have. What I can do now is (a) write down the metric function so it's ready the moment an observed outcome exists, and (b) check, as a baseline sanity step, how populated the top of today's queue actually is — a queue built on rows with barely any of the three signals observed would be a red flag before precision@K is even in play.

In [9]:
def precision_at_k(is_worth_refreshing, scores, k=20):
    """is_worth_refreshing: OBSERVED post-refresh outcome (not available yet - see Section 2).
    scores: the priority score used to rank candidates.
    Returns the fraction of the top-k scored items that were actually worth refreshing."""
    order = np.argsort(-np.asarray(scores))[:k]
    return np.asarray(is_worth_refreshing)[order].mean()

# Can't call precision_at_k for real yet (no observed outcome column). What we CAN check today:
K = 20
this_weeks_queue = lane.sort_values('opportunity_score', ascending=False).head(K)
print(f'Signals observed per row, top {K} candidates by proxy score:')
print(this_weeks_queue['n_signals_observed'].value_counts().sort_index(ascending=False))
print()
print(f'Same check over all {len(lane):,} rows (sanity baseline):')
print(lane['n_signals_observed'].value_counts().sort_index(ascending=False))

Signals observed per row, top 20 candidates by proxy score:
n_signals_observed
3     9
2    11
Name: count, dtype: int64

Same check over all 28,795 rows (sanity baseline):
n_signals_observed
3    11759
2    17036
Name: count, dtype: int64


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

**One row = one pseudonymized content item.** `content_id` is unique across the raw 30,000-row file; within this lane (after Section 2 drops the 1,205 rows with no position data) 28,795 remain, still one row per item — confirmed below — carrying its trailing-90-day metrics, metadata, and the proxy opportunity signals from Section 2. Sorting by `opportunity_score` and looking at the top of the queue is the actual deliverable an editor would see.

In [10]:
assert lane['content_id'].is_unique, 'expected one row per content_id'
print('one row = one content item:', lane['content_id'].is_unique, f"({lane['content_id'].nunique():,} unique)")

cols = ['content_id', 'client_id', 'content_type', 'avg_position', 'days_since_last_update',
        'clicks_prev_30d', 'clicks_last_30d', 'recent_pct_change', 'climb_room',
        'search_volume', 'cpc', 'opportunity_score', 'n_signals_observed']

preview = lane.sort_values('opportunity_score', ascending=False)[cols].head(10)
preview

one row = one content item: True (28,795 unique)


,content_id,client_id,content_type,avg_position,days_since_last_update,clicks_prev_30d,clicks_last_30d,recent_pct_change,climb_room,search_volume,cpc,opportunity_score,n_signals_observed
12156,content_8f21bfd738e7,client_d4735e3a26,keyword article,5.5,211,1,0,-1.0,1.0,NaN,NaN,0.856168,3
23140,content_6c61fc88189c,client_d4735e3a26,keyword article,5.3,211,1,0,-1.0,1.0,NaN,NaN,0.856168,3
25303,content_a0fa8bac2adc,client_d4735e3a26,keyword article,6.4,211,2,0,-1.0,1.0,NaN,NaN,0.856168,3
520,content_7749113e016d,client_9f14025af0,keyword article,11.8,151,1,0,-1.0,1.0,10.0,0.00,0.854941,3
25925,content_108e8e4899b7,client_9f14025af0,keyword article,13.3,151,1,0,-1.0,1.0,10.0,7.67,0.854941,3
25801,content_bef9eb4949af,client_7f2253d7e2,keyword article,6.4,106,1,0,-1.0,1.0,0.0,0.00,0.854107,3
2574,content_fd93c477fff0,client_f369cb89fc,keyword article,6.6,106,1,0,-1.0,1.0,10.0,0.00,0.854107,3
6424,content_a8680d500da8,client_7f2253d7e2,keyword article,9.7,106,1,0,-1.0,1.0,0.0,0.00,0.854107,3
5801,content_1d4f0e0d7b93,client_f74efabef1,keyword article,7.8,105,1,0,-1.0,1.0,10.0,0.00,0.853598,3
26242,content_55a5b1c46474,client_4ec9599fc2,keyword article,7.5,373,0,0,NaN,1.0,0.0,0.00,0.842151,2


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

**Because the signals that make something "worth refreshing" don't agree with each other.**

The table below correlates my three raw ingredients against each other and against the data's own `trend_pct` field. If a single hand-written rule (say, "flag anything with `trend_pct` below -20%") were enough, we'd expect it to line up with the other reasonable decline/opportunity signals. It doesn't: `trend_pct` and my independently-built `recent_pct_change` (last-30 vs. prev-30 days, computed straight from the raw click counts) correlate only weakly — two sensible measures of "is this declining" that mostly disagree on *which specific items* are declining. Climb-room and staleness are close to uncorrelated with either.

That's the concrete version of the framing skill's justification: several real, weak signals that disagree with each other, across 32 clients with likely different baselines, on 30,000 items today (and 500K+ in the full warehouse). Hand-tuning weights across that many disagreeing, heterogeneous signals is exactly the case a plain if-statement can't hold — but a model that learns how to weigh them can.

In [11]:
corr_cols = ['recent_pct_change', 'climb_room', 'staleness', 'trend_pct']
corr = lane[corr_cols].corr()
print('correlation between candidate signals:')
display(corr.round(2))

r = lane[['recent_pct_change', 'trend_pct']].corr().iloc[0, 1]
print(f"\nrecent_pct_change vs trend_pct: r = {r:.2f} - two reasonable 'is this declining?' "
      f"measures that mostly disagree on which items are declining.")

correlation between candidate signals:


,recent_pct_change,climb_room,staleness,trend_pct
recent_pct_change,1.00,-0.01,0.03,0.09
climb_room,-0.01,1.00,-0.06,-0.02
staleness,0.03,-0.06,1.00,-0.01
trend_pct,0.09,-0.02,-0.01,1.00



recent_pct_change vs trend_pct: r = 0.09 - two reasonable 'is this declining?' measures that mostly disagree on which items are declining.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.